<a href="https://colab.research.google.com/github/chamudithamk/Data-Driven-Dynamics-and-Control/blob/main/Dynamical_Systems_with_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dynamical Systems with Machine Learning**

### **Overview of Dynamical Systems**

Traditionally, the dynamics of a system has been derived using Newtonian, Lagrangian or Hamiltonian first principles.

General form of a dynamical equation:

$$\dot{x} = f(x,t,u; \beta) + d$$

where $d$ is disturbance and $\beta$ is a system paramter that changes dynamic behaviour of a system (e.g., altitude of a aeroplane).

***NOTE**: $\beta$ can be used as a design parameter to control a system whereas $u$ is typically used in real-time feedback controllers*

However, first princples are only models of reality, sometimes they are extremely good and sometimes, $f$ is simply unknown. This is where data-driven approaches come into play. Data-driven approaches can tackle a number of other complexities as well: nonlinearity, uncertainty, high-dimensionality, stochasticity, chaos (highly sensitive to initial conditions and other parameters), transients, multi-scale (e.g., turbulence scales).

The measurements of such a dynamical system:

$$y = g(x,t) + n$$

where $n$ is noise. $y$ is not necessarily full-state and is estimated through an observer (e.g., Kalman Filter).

Ultimately, they can be used to predict future states, control and for design and optimisation purposes. Also, it is important that ML models are generalisable and interpretable rather than black-box models.

#### Example: Simulation of Lorenz Equations

It is a set of three coupled nonlinear ordinary differential equations founded by Edward Lorenz in 1963 to show why long-range weather forecasting is fundamentally limited. It is the founding example of chaos theory and is a standard test problem for numerical solvers, data assimilation and ML time-series forecasting.

$$\dot{x} = \sigma(y-x)$$
$$\dot{y} = x(\rho -z)-y$$
$$\dot{z} = xy-\beta z$$

State-space form of this can be given by:

$$\dot{\mathbf{x}} = f(\mathbf{x},t; \mathbf{b})$$

where

$$\mathbf{x} = \begin{bmatrix}
x \\ y \\ z
\end{bmatrix}; \;\;\;\; \mathbf{b} = \begin{bmatrix} \sigma \\ \rho \\ \beta \end{bmatrix}$$



In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import solve_ivp

# ---------------- INPUTS ----------------
b  = [10.0, 28.0, 8/3]     # [sigma, rho, beta]
x0 = [1.0, 1.0, 1.0]       # initial state [x, y, z]
dt = 0.01                  # output time step
T  = 50.0                  # total duration
method = "RK45"            # "RK45", "DOP853", "LSODA", "Radau", "BDF"
# ----------------------------------------

In [ ]:
def lorenz(t, state, sigma, rho, beta):
    x, y, z = state
    return [sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z]

t_eval = np.arange(0, T + dt / 2, dt)

sol = solve_ivp(lorenz, (0, T), x0, args=tuple(b),
                method=method, t_eval=t_eval,
                rtol=1e-9, atol=1e-12)
t = sol.t
x, y, z = sol.y

# --------- 3D Lorenz butterfly attractor visualisation ---------
fig3d = go.Figure(go.Scatter3d(
    x=x, y=y, z=z, mode="lines",
    line=dict(width=3, color=t, colorscale="Viridis",
              colorbar=dict(title="t"))
))
fig3d.update_layout(
    title=f"Lorenz attractor",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    height=650
)
fig3d.show()
# ----------------------------------------------------------------

**What is Chaos?**
Small differences in initial conditions can grow until the resulting trajectories become significantly different.

**Why does a continous-time system need at least 3-dimensions for Chaos to occur?**

The fundamental rule is that two trajectories cannot cross, if it does, it means it has the same initial condition at that point. In 1D, the state cannot reverse direction arbitrarily (it either converges or diverges); in 2D, the Poincaré–Bendixson theorem (if a state is in a bounded region, it either converges or in a limit cycle) rules out chaotic attractors. At least 3 state dimensions are needed for this kind of continuous-time chaos, but how?

Suppose two points start close together. As the system evolves, stretching causes the points to move farther apart, making the system sensitive to initial conditions. Meanwhile, the deterministic dynamics of the system cause the space to fold back on itself, bringing different regions into the same region. When stretching and folding occur repeatedly, they can produce chaotic dynamics, as observed in the Lorenz (butterfly) attractor.



#### Discrete-Time Systems

Dynamic equation for a continous autonomous system:

$$\dot{x} = f(x)$$

A discrete-time system, however, evolves in steps.

$$x_{k+1} = F(x_k)$$

If $\Delta t$ is the time-step of an integrator, then the discrete-time dynamical system can be defined by:

$$x_k = x(k \cdot \Delta t)$$

Then, the flow map $F_{\Delta t}$ of states is given by:

$$x_{k+1} = x_k + \int_{k\Delta t}^{(k+1)\Delta t} f(x(\tau)) dt$$

In most dynamical systems of interest, computation of flow map $F_{\Delta t}$ solution in closed-form is not possible and must be adapted to certain numerical schemes such as Forward Euler, Runge-Kutta, etc.

#### Example: Logistic Map Discrete-Time System

It is a population dynamic model popularised by biologist Robert May in 1976, which shows trivially simple nonlinear rule can produce extremely complicated behaviour.

$$x_{k+1}=\beta x_k (1-x_k)$$

where $\beta$ is the growth rate of population $x_k$. Due to finite Earth resources, term $(1-x_k)$ is introduced as the crowding that limits the growth.

Let us demonstrate this system's period-doubling route to chaos as $\beta$ increases through its bifurcation diagram.

In [ ]:
betas = np.linspace(0, 4.0, 500)
n_transient, n_keep = 500, 200

x = 0.5 * np.ones_like(betas)
for _ in range(n_transient):
    x = betas * x * (1 - x)

B, X = [], []
for _ in range(n_keep):
    x = betas * x * (1 - x)
    B.append(betas.copy()); X.append(x.copy())

fig = go.Figure(go.Scattergl(
    x=np.concatenate(B), y=np.concatenate(X),
    mode="markers", marker=dict(size=1.5, color="black", opacity=0.3)))
fig.update_layout(title="Logistic map bifurcation diagram",
                  xaxis_title="β", yaxis_title="x (steady state)", height=600)
fig.show()

From the above bifurcation diagram, it can be seen that:

<center>
<table border="1">
  <tr align="center">
    <td><b>Parameter Range</b></td>
    <td><b>Population Behavior</b></td>
    <td><b>Dynamic State</b></td>
  </tr>

  <tr align="center">
    <td>&beta; &le; 1</td>
    <td>Population dies out i.e. x &rarr; 0</td>
    <td>Extinction</td>
  </tr>
  <tr align="center">
    <td>1 &lt; &beta; &le; 3</td>
    <td>Population stabilises to a finite value</td>
    <td>Stable Equilibrium</td>
  </tr>
  <tr align="center">
    <td>3 &lt; &beta; &le; 3.45</td>
    <td>Population oscillates between two values after first bifurcation</td>
    <td>Period-2 Orbit</td>
  </tr>
  <tr align="center">
    <td>3.45 &lt; &beta; &le; 3.54</td>
    <td>Population oscillates between four values after second bifurcation</td>
    <td>Period-4 Orbit</td>
  </tr>
  <tr align="center">
    <td>3.54 &lt; &beta; &le; 3.84</td>
    <td>Chaotic population wanders without repeating</td>
    <td>Chaos</td>
  </tr>
  <tr align="center">
    <td>&beta; &approx; 3.84</td>
    <td>Population oscillates between three values</td>
    <td>Order Inside Chaos</td>
  </tr>
  <tr align="center">
    <td>3.84 &lt; &beta; &le; 4</td>
    <td>Fully developed chaos</td>
    <td>Complete Chaos</td>
  </tr>
  <tr align="center">
    <td>&beta; &gt; 4</td>
    <td>Population escapes [0,1]</td>
    <td>Divergence</td>
  </tr>
</table>
</center>

**How can a 1-dimensional discrete-time system turn into chaos?**

In discrete systems, chaos can occur even with 1 state variable, as in the logistic map. Here, due to the boundedness $[0,1]$ of the dynamic equation of logistic map, and the ability to jump between states $x_k$ and $x_{k+1}$,


*   Two nearby initial conditions can stretch apart, as in continuous-time systems.
*   The map can also fold, mapping different states back into the same region and reversing their order without necessarily revisiting the same state.

When stretching and folding occur repeatedly, chaotic behaviour can emerge. In a one-dimensional autonomous continuous-time system, trajectories cannot reverse direction or cross one another, since doing so would violate uniqueness of solutions.



### **Dynamic Mode Decomposition**

Dynamic Mode Decomposition (DMD) is a data-driven (no governing equations required) modal decomposition where each mode consists of spatially correlated structures that have the same linear
behavior in time (e.g., oscillations at a given frequency with growth or decay).

First, a sufficient number ($m$) of snapshots of data (states) must be collected $\{ x_1, x_2, \dots, x_m \} \in \mathbb{R}^{\text{n}}$ as the dynamics evolve in time. Time-steps $\Delta t$ must be sufficiently small to resolve the highest frequencies in the dynamics.

Next, organise them into matrices $\mathbf{X}, \mathbf{X} \in \mathbb{M}_{\text{n}\times \text{m}}$ such that:

$$\mathbf{X} = \begin{bmatrix}
  \mid & \mid & & \mid & \mid \\
  x(t_1) & x(t_2) & \dots & x(t_{m-2}) & x(t_{m-1})  \\
  \mid & \mid & & \mid & \mid
\end{bmatrix}$$

$$\mathbf{X}' = \begin{bmatrix}
  \mid & \mid & & \mid & \mid \\
  x(t_2) & x(t_3) & \dots & x(t_{m-1}) & x(t_{m})  \\
  \mid & \mid & & \mid & \mid
\end{bmatrix}$$

DMD algorithm then seeks a best-fit linear operator $\mathbf{A}$ (regression) that advances $\mathbf{X}$ to $\mathbf{X}'$ in time.

$$\mathbf{X}' \approx \mathbf{A} \mathbf{X}$$

If we assume uniform sampling in time,

$$x_k = x(k \cdot \Delta t)$$
$$x_{k+1} \approx \mathbf{A}x_k$$

Mathematically, best-fit linear operator $\mathbf{A}$ is:
$$\mathbf{A} = \mathbf{X}'\mathbf{X}^\dagger$$

We are often interested in eigenvectors $\Phi$ and eigenvalues $\Lambda$ of $\mathbf{A}$.

$$\mathbf{A}\Phi = \Phi \Lambda $$

However, for a high-dimensional state vector $x \in \mathbb{R}^{\text{n}}$, the matrix $\mathbf{A}$ has $n^2$ elements, and representing this operator, let alone computing its spectral decomposition, may be intractable.

Instead, the DMD algorithm leverages dimensionality reduction (SVD) to compute the dominant eigenvalues and eigenvectors of $\mathbf{A}$ without requiring any explicit computations using $\mathbf{A}$ directly.

Since typically $m \ll n$, DMD algorithm projects $\mathbf{A}$, which has rank $m$ at most, onto its leading $m$ singular vectors. This results in a small matrix $\tilde{\mathbf{A}} \in \mathbb{M}_{\text{m}\times\text{m}}$ with $m^2$ elements.